# Módulo 5 · Ingesta de múltiples archivos → Bronze

⏱ **10 min** · Slides: 27–28 (los 4 pasos de la ingesta y la capa Bronze)

**Objetivo:** leer todos los JSON de RAW con **un solo comando** y guardarlos en una tabla Delta que sepa de qué archivo viene cada fila.

| Slide 27 | Paso del laboratorio |
|---|---|
| 1. Define el esquema | Paso 1 |
| 2. Lee con comodines | Paso 2 |
| 3. Registra el origen | Paso 3 |
| 4. Escribe en Delta (append) | Paso 4 |
| Verificar | Paso 5 |

**Bronze** = dato crudo, tal como llega, más el archivo de origen. No se limpia nada todavía.

**Requiere:** módulo 2.

In [0]:
%run ./00_Setup

## Paso 0 · Empezar limpio
Borramos la tabla para poder repetir el módulo sin acumular datos.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}")

## Paso 1 · Define el esquema
Ya está en `ESQUEMA_EPISODIOS` (00_Setup). No se infiere en cada carga.

In [0]:
print(ESQUEMA_EPISODIOS)

## Paso 2 · Lee con comodines
El patrón `*/*/*/tvmaze.json` equivale a "año/mes/día" de cualquier fecha.

In [0]:
print("Patrón:", RAW_GLOB)

df = (spark.read.format("json")
      .option("multiLine", True)
      .schema(ESQUEMA_EPISODIOS)
      .load(RAW_GLOB))

print("Filas leídas:", df.count())

## Paso 3 · Registra el origen

`_metadata` es una columna oculta que Spark ofrece al leer archivos. ¿`file_name` o `file_path`?

In [0]:
meta = df.select("_metadata.file_name", "_metadata.file_path")

print("Valores distintos de file_name:", meta.select("file_name").distinct().count())
print("Valores distintos de file_path:", meta.select("file_path").distinct().count())

meta.dsiplay()

👀 **Qué observar:** todos los archivos se llaman `tvmaze.json`, así que `file_name` no distingue nada.
Usamos **`file_path`**, que incluye año/mes/día.

In [0]:
df_bronze = df.select(
    "*",
    F.col("_metadata.file_path").alias("archivo_origen"),
    F.current_timestamp().alias("ingestado_en"),
)

df_bronze.display()

## Paso 4 · Escribe en Delta con `append`

In [0]:
(df_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(BRONZE))

print(f"✔ Tabla creada: {qualified_name(BRONZE)}")

## Paso 5 · Verificar

In [0]:
bronze = spark.table(BRONZE)
print("Filas en Bronze:", bronze.count())

In [0]:
# Filas por archivo de origen: debe haber un grupo por cada día descargado
display(bronze.groupBy("archivo_origen").count().orderBy("archivo_origen"))

In [0]:
# Control básico de calidad: ninguna fila debería tener id nulo
nulos = bronze.filter(F.col("id").isNull()).count()
print("Filas con id nulo:", nulos)

In [0]:
# Bronze conserva la estructura anidada del JSON
display(bronze.select("id", "name", "airdate", "_embedded.show.name", "archivo_origen").limit(5))

In [0]:
# La tabla es Delta: tiene formato, archivos e historial
display(spark.sql(f"DESCRIBE DETAIL {BRONZE}").select("format", "numFiles", "sizeInBytes"))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {BRONZE}").select("version", "operation", "operationMetrics"))

👀 **Qué observar:** una fila por episodio, agrupadas por archivo; el campo `_embedded` sigue anidado;
y la tabla tiene historial (versión 0 = esta carga). Búscala en **Catalog → lab_tvmaze → bronze_episodios**.

## ✔ Checkpoint

- [ ] `bronze_episodios` existe y su conteo coincide con el del Paso 2
- [ ] Sé por qué usamos `file_path` y no `file_name`
- [ ] Id nulos = 0

⚠️ **Pregunta para el próximo módulo:** ¿qué pasa si ejecuto el Paso 4 dos veces?

**Siguiente:** `06_Problemas_de_ingesta`